<a href="https://colab.research.google.com/github/rohitblpprajapat/100-days-of-code/blob/master/continual_pretraining_of_llama_3_2_1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pprint import pprint
import math
import wandb

import datasets
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import DataCollatorForLanguageModeling
from transformers import TrainingArguments, Trainer
from huggingface_hub import login

In [2]:
wandb.init(
    project="DLP-w4-cpt-node-1",
    config={
        "batch_size":4,
        "dataset":"Sangraha"

    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 22f1001536 (22f1001536-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
login()

In [4]:
ds = load_dataset("ai4bharat/sangraha", data_files="https://huggingface.co/datasets/ai4bharat/sangraha/resolve/main/verified/tam/data-0.parquet")

README.md: 0.00B [00:00, ?B/s]

verified/tam/data-0.parquet:   0%|          | 0.00/358M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'text', 'type'],
        num_rows: 149796
    })
})

In [6]:
model_id = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Context length: {tokenizer.model_max_length}')

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Vocab size: 128000
Context length: 131072


In [7]:
tokenizer.model_max_length = 1024
tokenizer.pad_token = tokenizer.eos_token

In [8]:
eg = ds['train'][1]
num_words = len(eg['text'].split())
print(num_words)

47


In [9]:
input_ids = tokenizer(eg['text'])['input_ids']
print(input_ids)
print(len(input_ids))

[128000, 20627, 248, 32601, 228, 20627, 107, 64500, 106, 84298, 20627, 109, 32601, 230, 20627, 225, 198, 20627, 103, 20627, 248, 64500, 248, 20627, 108, 100112, 248, 91702, 71697, 106, 20627, 109, 64500, 109, 84298, 20627, 106, 47454, 71697, 103, 20627, 248, 64500, 248, 32601, 230, 20627, 103, 64500, 103, 20627, 107, 20627, 109, 32601, 230, 71697, 240, 20627, 102, 64500, 109, 20627, 122, 20627, 243, 71697, 248, 32601, 229, 20627, 108, 64500, 97, 64500, 97, 84298, 71697, 240, 20627, 108, 84298, 71697, 106, 20627, 96, 91702, 71697, 101, 32601, 229, 20627, 108, 20627, 106, 47454, 71697, 232, 20627, 109, 71697, 113, 32601, 230, 20627, 243, 64500, 243, 20627, 113, 84298, 20627, 106, 47454, 13, 71697, 232, 20627, 109, 100112, 107, 71697, 227, 20627, 108, 100112, 248, 91702, 11, 71697, 103, 20627, 107, 20627, 109, 84298, 20627, 253, 20627, 102, 47454, 11, 71697, 97, 32601, 229, 20627, 247, 64500, 243, 20627, 122, 20627, 107, 47454, 71697, 97, 84298, 20627, 108, 84298, 20627, 113, 20627, 110, 

In [10]:
print(f'The fertility rate is: {len(input_ids)/num_words}')

The fertility rate is: 11.085106382978724


In [12]:
model = AutoModelForCausalLM.from_pretrained(model_id, pad_token_id =tokenizer.eos_token_id )

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [13]:
configuration = model.config
print(configuration)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 128001,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 128256
}



In [14]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128001)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)